# Straight line vs road: why travel times need OSRM

A field officer rides a two-wheeler on real roads, not in a straight line ("as the crow flies").
This notebook takes a few real Baramati villages and compares, for a trip from the branch:

* **straight-line distance** — the haversine formula on the two coordinates;
* **road distance and time** — from the public **OSRM** server (the same service the app uses).

Run it top to bottom (*Run → Run All*); it needs internet access. With the project venv activated:

```
pip install -r requirements-data.txt
python -m jupyter notebook scripts/notebooks/travel_time_comparison.ipynb
```

(`python -m jupyter` makes sure the venv's Jupyter is used. In VS Code, pick the `.venv` interpreter as the kernel.)

In [1]:
import math
import time
from pathlib import Path

import folium
import pandas as pd
import requests

OSRM_URL = "https://router.project-osrm.org"
BRANCH = {"name": "Baramati Branch", "lat": 18.151, "lng": 74.577}

# The notebook lives in scripts/notebooks/; the data is two folders up.
villages = pd.read_csv(Path("../../data/villages_baramati.csv"), dtype={"census_code": "string"})
print(f"{len(villages)} villages loaded")
villages.head()

117 villages loaded


,name,lat,lng,census_code,population,match_score,coord_source
0,Ambi Bk,18.287557,74.277863,556876,1184.0,NaN,census_boundary
1,Ambi Kh,18.318109,74.286478,556875,1091.0,NaN,census_boundary
2,Anjangaon,18.223447,74.502432,556912,1755.0,NaN,census_boundary
3,Baburdi,18.274520,74.377898,556883,1889.0,100.0,osm
4,Bajrangwadi,18.089409,74.399104,556960,879.0,NaN,census_boundary


## 1. Distances for sample trips

`haversine_km` is the straight-line distance on the Earth's surface. OSRM's **Route API** returns the road
distance and a **car** travel time; the app multiplies car times by a *two-wheeler speed factor* (default 1.0).

In [2]:
SAMPLE_VILLAGES = ["Supe", "Morgaon", "Malegaon Bk", "Vadgaon Nimbalkar", "Katphal", "Jalgaon Supe"]


def haversine_km(lat1, lng1, lat2, lng2):
    r = 6371.0
    d_lat, d_lng = math.radians(lat2 - lat1), math.radians(lng2 - lng1)
    a = math.sin(d_lat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(d_lng / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def osrm_route(a, b):
    """Road route from a to b: (km, car minutes, [lat, lng] points of the road line)."""
    url = f"{OSRM_URL}/route/v1/driving/{a['lng']},{a['lat']};{b['lng']},{b['lat']}?overview=full&geometries=geojson"
    route = requests.get(url, headers={"User-Agent": "MFI-Route-Optimizer notebook"}, timeout=30).json()["routes"][0]
    line = [[lat, lng] for lng, lat in route["geometry"]["coordinates"]]
    return route["distance"] / 1000, route["duration"] / 60, line


rows, lines = [], {}
for name in SAMPLE_VILLAGES:
    v = villages[villages["name"] == name].iloc[0]
    straight = haversine_km(BRANCH["lat"], BRANCH["lng"], v["lat"], v["lng"])
    road_km, car_min, line = osrm_route(BRANCH, v)
    lines[name] = line
    rows.append({
        "village": name,
        "straight_km": round(straight, 1),
        "road_km": round(road_km, 1),
        "detour_factor": round(road_km / straight, 2),
        "car_min": round(car_min),
        "avg_road_kmh": round(road_km / car_min * 60),
    })
    time.sleep(1)  # the public OSRM server allows about one request per second

trips = pd.DataFrame(rows)
trips

,village,straight_km,road_km,detour_factor,car_min,avg_road_kmh
0,Supe,29.4,33.9,1.16,33,62
1,Morgaon,30.9,33.1,1.07,31,63
2,Malegaon Bk,7.3,8.0,1.09,8,61
3,Vadgaon Nimbalkar,22.6,25.3,1.12,25,61
4,Katphal,10.5,13.6,1.29,15,54
5,Jalgaon Supe,15.5,17.2,1.11,17,60


In [3]:
print(f"Average detour factor: {trips['detour_factor'].mean():.2f}× — roads are that much longer than straight lines.")
print(f"Largest: {trips.loc[trips['detour_factor'].idxmax(), 'village']} ({trips['detour_factor'].max():.2f}×)")
print(f"Average road speed ranges from {trips['avg_road_kmh'].min()} to {trips['avg_road_kmh'].max()} km/h depending on the roads used.")

Average detour factor: 1.14× — roads are that much longer than straight lines.
Largest: Katphal (1.29×)
Average road speed ranges from 54 to 63 km/h depending on the roads used.


## 2. On the map

Grey dots are all villages. For each sample village the **dashed line** is the straight line and the
**solid line** is the road OSRM would take.

In [4]:
m = folium.Map(location=[BRANCH["lat"], BRANCH["lng"]], zoom_start=11, tiles="OpenStreetMap")

for v in villages.itertuples():
    folium.CircleMarker([v.lat, v.lng], radius=3, color="#64748b", fill=True, tooltip=v.name).add_to(m)

folium.Marker([BRANCH["lat"], BRANCH["lng"]], tooltip=BRANCH["name"], icon=folium.Icon(color="red", icon="home")).add_to(m)

colors = ["#2563eb", "#16a34a", "#9333ea", "#ea580c", "#0891b2", "#db2777"]
for color, trip in zip(colors, trips.itertuples()):
    v = villages[villages["name"] == trip.village].iloc[0]
    folium.PolyLine([[BRANCH["lat"], BRANCH["lng"]], [v["lat"], v["lng"]]], color=color, weight=2, dash_array="6 6",
                    tooltip=f"{trip.village}: straight {trip.straight_km} km").add_to(m)
    folium.PolyLine(lines[trip.village], color=color, weight=4,
                    tooltip=f"{trip.village}: road {trip.road_km} km, {trip.car_min} min").add_to(m)

m

## 3. What this means for route planning

* Road distances are clearly longer than straight lines, and the extra length varies from village to
  village (rivers, canals and the road network decide it). A fixed multiplier on straight-line distance
  would be wrong for some villages.
* Travel **time** also depends on the kind of road, which straight lines cannot capture.
* So the optimizer uses a **road travel-time matrix** from OSRM (`lib/travel/`): every pair of points
  (officer start locations and meeting places) is asked once, stored in the `travel_cache` table, and
  reused. The two-wheeler **speed factor** is applied at solve time, so changing it never needs new OSRM calls.